In [29]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader



import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from Data.class_dataset import MRIDataset
from Model.model import build_vit3d

# --- Cargar datos de test ---

df=pd.read_csv("../Training/training_data.csv")

_, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
test_imgs = test_df["Path"].tolist()
test_ages = test_df["Age"].tolist()
test_dataset = MRIDataset(test_imgs, test_ages)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# --- Cargar modelo ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_vit3d()
state_dict = torch.load("../Training/Trained_models/model_4error.pth", map_location=device)
# Si las claves tienen 'module.' al inicio, elimínalo
if any(k.startswith('module.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace('module.', '', 1)
        new_state_dict[new_key] = v
    state_dict = new_state_dict
model.load_state_dict(state_dict)
model.to(device)
model.eval()



ViT3D(
  (patch_to_embedding): Linear(in_features=4096, out_features=1024, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=1024, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=1024, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=1024, out_features=2048, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=

In [30]:
# --- Evaluar ---
all_preds = []
all_ages = []
with torch.no_grad():
    for imgs, ages in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        all_preds.append(preds.cpu())
        all_ages.append(ages.unsqueeze(1).cpu())
all_preds = torch.cat(all_preds).numpy().flatten()
all_ages = torch.cat(all_ages).numpy().flatten()
all_paths = test_df["Path"].tolist()

mae = mean_absolute_error(all_ages, all_preds)
r2 = r2_score(all_ages, all_preds)
print(f"Test MAE: {mae:.2f}")
print(f"Test R2: {r2:.2f}")

Test MAE: 5.42
Test R2: 0.91


In [31]:
#crear un df con las edades reales, predichas y el ID
results_df = pd.DataFrame({
    "ID": all_paths,
    "Age": all_ages,
    "Prediction": all_preds,
    "Error": all_preds - all_ages, 
    'Absolute Error': np.abs(all_preds - all_ages)
})
results_df['ID']=results_df['ID'].replace('/data/lautaro/quasiraw/', '', regex=True)
results_df['ID']=results_df['ID'].replace('.nii.gz', '', regex=True)
results_df.sort_values(by='Absolute Error', ascending=True, inplace=True)
results_df.to_csv("test_predictions_4error.csv", index=False)

In [32]:
results_df

,ID,Age,Prediction,Error,Absolute Error
614,sub-OAS31324_sess-d0218_run-02,74.519997,74.552948,0.032951,0.032951
43,sub-OAS30993_ses-d0059,70.180000,70.217712,0.037712,0.037712
236,sub-OAS30241_ses-d1203_run-02,50.930000,50.890232,-0.039768,0.039768
625,sub-173601103742,21.000000,21.046148,0.046148,0.046148
375,sub-200367335302,23.000000,22.936678,-0.063322,0.063322
...,...,...,...,...,...
493,sub-OAS31285_sess-d0055,56.169998,84.305054,28.135056,28.135056
514,sub-561752414596,46.000000,17.410561,-28.589439,28.589439
155,sub-CC310142,48.000000,76.736908,28.736908,28.736908
273,sub-846761410374,49.000000,17.276493,-31.723507,31.723507
